# Robustness check: applied lemma reduction vs. raw suffixes

**Purpose**: Test whether the **time trends of relative distributions** remain stable
when comparing the actually applied reduction from `suffix_lemma` with the raw suffixes without lemmatization.

**Period**: 2022-04-21 to 2025-02-08 (scraper cutoff to last crawled date; update 2026-03-20)

**Applied reduction (as used in processing and in notebook 06)**:
- Klimawandel: `suffix_lemma == wandel`
- Klimakrise: `suffix_lemma == krise`
- Klimaschutz: `suffix_lemma == schutz`

**Comparison variant (raw, no lemma)**:
- Klimawandel: `suffix in {wandel, wandels}`
- Klimakrise: `suffix in {krise, krisen}`
- Klimaschutz: `suffix in {schutz, schutzes}`

## Setup: imports and data loading

In [ ]:
import os
import sys
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# phinx: Sphinx documentation tool.

notebook_dir = (
    os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
)
sys.path.append(os.path.join(notebook_dir, "..", "pylib"))

from analysis_utils import map_suffix_to_term
from config import DEFAULT_END_DATE, DWH_DB_PATH, PLOTS_DIR, SCRAPER_CUTOFF
from handle_sqlite import read_table_as_dataframe
from plotting_utils import apply_plot_style

apply_plot_style()

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
plot_dir = PLOTS_DIR

# Load processed tables from SQLite
df_context = read_table_as_dataframe("context_processed", str(DWH_DB_PATH))
df_meta = read_table_as_dataframe("newspapers", str(DWH_DB_PATH))

print(f"Context rows: {len(df_context)}")
print(f"Metadata rows: {len(df_meta)}")
print(f"Context columns: {df_context.columns.tolist()}")

In [ ]:
df_meta['data_published'] = pd.to_datetime(df_meta['data_published'])

We keep a left join on metadata: the ~17,004 left-join rows are metadata entries without a matching context row.
That can mean there was no "Klima" hit for that newspaper/day, or the context extraction produced no row.
Because we want to keep the option to show newspaper-days without Klima mentions, we retain them.
We also have more context entries because we apply a date cutoff, so rows outside the window are dropped by the join.


In [ ]:
analysis_start = SCRAPER_CUTOFF
analysis_end = DEFAULT_END_DATE

df_meta_filtered = df_meta[df_meta["data_published"] >= analysis_start].copy()
if analysis_end is not None:
    df_meta_filtered = df_meta_filtered[df_meta_filtered["data_published"] <= analysis_end].copy()

analysis_end_label = analysis_end.date() if analysis_end is not None else "open end"
print(f"Total metadata rows: {len(df_meta)}")
print(f"Metadata rows from {analysis_start.date()} to {analysis_end_label}: {len(df_meta_filtered)}")

In [ ]:
# Check for null values in key columns before inner join
print("=== NULL VALUES IN df_context ===")
print(df_context.isnull().sum())
print(f"\nTotal rows in df_context: {len(df_context)}")

print("\n=== NULL VALUES IN df_meta_filtered ===")
print(df_meta_filtered.isnull().sum())
print(f"\nTotal rows in df_meta_filtered: {len(df_meta_filtered)}")

# Show what will be lost in inner join
print("\n=== IMPACT OF INNER JOIN ===")
print(f"Rows in df_meta_filtered: {len(df_meta_filtered)}")
print(f"Rows in df_context (before filter): {len(df_context)}")

# Check how many context rows have matching newspaper_ids in df_meta_filtered
matching_ids = df_context['newspaper_id'].isin(df_meta_filtered['newspaper_id']).sum()
print(f"Context rows with matching newspaper_id in df_meta_filtered: {matching_ids}")
print(f"Context rows that will be lost: {len(df_context) - matching_ids}")

In [ ]:
# Merge: meta is left → all meta rows kept, context joined where available
df = df_meta_filtered[['newspaper_id', 'newspaper_name', 'data_published']].merge(df_context, on='newspaper_id', how='left')

print(f"Totale Einträge von {SCRAPER_CUTOFF}: {len(df)} Nennungen")

In [ ]:
print(f"Zeitungen: {df['newspaper_name'].nunique()}")
print(f"\nSpalten in df: {df.columns.tolist()}")

## Comparison: applied reduction vs. raw exact suffixes

In [ ]:
# Applied mapping after processing / notebook 06
LEMMA_TO_TERM = {
    "wandel": "Klimawandel",
    "krise": "Klimakrise",
    "schutz": "Klimaschutz",
}

df["term"] = (
    df["suffix_lemma"]
    .str.lower()
    .str.strip()
    .map(LEMMA_TO_TERM)
)

print("\nApplied lemma targets:")
for suffix, term in LEMMA_TO_TERM.items():
    print(f"- {term}: {suffix}")

df_reduced = df[df["term"].notna()].copy()
print(f"Mentions of the three terms after applied reduction: {len(df_reduced)}")
print(df_reduced["term"].value_counts())

## Quarterly comparison: relative shares and trend similarity

In [ ]:
# Raw comparison: exact suffix list without lemma reduction
RAW_SUFFIX_MAP = {
    "wandel": "Klimawandel",
    "wandels": "Klimawandel",
    "krise": "Klimakrise",
    "krisen": "Klimakrise",
    "schutz": "Klimaschutz",
    "schutzes": "Klimaschutz",
}

df["suffix_clean"] = df["suffix"].str.lower().str.strip()
df_raw = df.copy()
df_raw["term"] = df_raw["suffix_clean"].apply(
    lambda value: map_suffix_to_term(value, RAW_SUFFIX_MAP)
 )
df_raw = df_raw[df_raw["term"].notna()].copy()

print("\n=== Raw comparison (exact suffixes, no lemma) ===")
print(f"Mentions: {len(df_raw)}")
print("\nDistribution:")
print(df_raw["term"].value_counts())
print("\nRaw suffix list:")
for suffix, term in RAW_SUFFIX_MAP.items():
    print(f"- {term}: {suffix}")

In [ ]:
# Show what lemma reduction adds beyond the raw exact suffix list
lemma_only = df_reduced[df_reduced["suffix"].str.lower().str.strip().isin(RAW_SUFFIX_MAP.keys()) == False].copy()

print("\n=== Extra raw suffix forms captured only via lemma reduction ===")
print(f"Mentions: {len(lemma_only)}")
print("\nDistribution after reduction:")
print(lemma_only["term"].value_counts())
print("\nMost frequent raw suffix variants added by lemma reduction (top 15):")
print(lemma_only["suffix"].str.lower().str.strip().value_counts().head(15))

## Figure: deviations with applied reduction vs. raw suffixes

In [ ]:
# Compare relative overall distributions (not absolute levels)
print("\n=== Relative overall distribution: applied reduction vs. raw exact suffixes ===")
print()

terms = ["Klimawandel", "Klimakrise", "Klimaschutz"]
reduced_counts = df_reduced["term"].value_counts().reindex(terms, fill_value=0)
raw_counts = df_raw["term"].value_counts().reindex(terms, fill_value=0)

reduced_shares_total = (reduced_counts / reduced_counts.sum() * 100).round(2)
raw_shares_total = (raw_counts / raw_counts.sum() * 100).round(2)

comparison = pd.DataFrame({
    "Applied reduction share (%)": reduced_shares_total,
    "Raw exact share (%)": raw_shares_total,
})
comparison["Diff (pp)"] = (comparison["Applied reduction share (%)"] - comparison["Raw exact share (%)"]).round(2)

print(comparison.to_string())
print()
print(f"Applied reduction mentions: {int(reduced_counts.sum())}")
print(f"Raw exact mentions: {int(raw_counts.sum())}")
print("Note: Relative shares are the focus of this robustness check.")

## Quarterly comparison: relative shares and trend similarity

In [ ]:
# Quarterly aggregation
reduced_copy = df_reduced.copy()
reduced_copy["quarter"] = reduced_copy["data_published"].dt.to_period("Q")
reduced_quarterly = reduced_copy.groupby(["quarter", "term"]).size().unstack(fill_value=0)

raw_copy = df_raw.copy()
raw_copy["quarter"] = raw_copy["data_published"].dt.to_period("Q")
raw_quarterly = raw_copy.groupby(["quarter", "term"]).size().unstack(fill_value=0)

# Align quarters/terms for fair comparison
quarter_index = reduced_quarterly.index.union(raw_quarterly.index)
term_index = ["Klimawandel", "Klimakrise", "Klimaschutz"]
reduced_quarterly = reduced_quarterly.reindex(index=quarter_index, columns=term_index, fill_value=0)
raw_quarterly = raw_quarterly.reindex(index=quarter_index, columns=term_index, fill_value=0)

# Relative shares per quarter
main_shares = (reduced_quarterly.div(reduced_quarterly.sum(axis=1), axis=0) * 100).fillna(0)
wild_shares = (raw_quarterly.div(raw_quarterly.sum(axis=1), axis=0) * 100).fillna(0)

print("\n=== Relative quarterly shares: applied reduction ===")
print(main_shares.round(2).to_string())
print("\n=== Relative quarterly shares: raw exact suffixes ===")
print(wild_shares.round(2).to_string())

# Trend similarity
share_diff = (wild_shares - main_shares).round(2)
mean_abs_diff = share_diff.abs().mean().round(2)
trend_corr = main_shares.corrwith(wild_shares).round(3)

print("\n=== Mean absolute deviation (pp) per term ===")
print(mean_abs_diff.to_string())
print("\n=== Correlation of quarterly share trends ===")
print(trend_corr.to_string())

## Figure: deviations with applied reduction vs. raw suffixes

In [ ]:
# Single-plot view with deviation segments
terms_plot = ["Klimaschutz", "Klimawandel", "Klimakrise"]
linestyles = {"Klimaschutz": "-", "Klimawandel": "--", "Klimakrise": ":"}
markers = {"Klimaschutz": "o", "Klimawandel": "s", "Klimakrise": "^"}

plot_index = main_shares.index
x = np.arange(len(plot_index))

fig, ax = plt.subplots(figsize=(12, 6))

for term in terms_plot:
    base = main_shares[term].reindex(plot_index).values
    wild = wild_shares[term].reindex(plot_index).values

    # Applied reduction
    ax.plot(
        x,
        base,
        linestyle=linestyles[term],
        marker=markers[term],
        color="black",
        linewidth=1.8,
        markersize=5,
        label=f"{term} (reduced)",
)

    # Raw exact suffixes
    ax.plot(
        x,
        wild,
        linestyle=linestyles[term],
        marker="x",
        color="dimgray",
        linewidth=1.0,
        markersize=5,
        alpha=0.8,
        label=f"{term} (raw)",
)

    # Vertical segments show deviations
    for xi, y0, y1 in zip(x, base, wild):
        ax.plot([xi, xi], [y0, y1], color="0.75", linewidth=0.8, alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels([str(q) for q in plot_index], rotation=45, ha="right")
ax.set_xlabel("Quarter")
ax.set_ylabel("Relative share (%)")
ax.set_title("Deviations with applied reduction vs. raw suffixes")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(loc="best", ncol=2, frameon=True)
ax.text(
    0.01,
    -0.2,
    "Gray segments = deviation between applied reduction and raw exact suffixes per quarter",
    transform=ax.transAxes,
    fontsize=9,
)

plt.tight_layout()
plot_path = os.path.join(plot_dir, "07_abweichungen_reduktion_vs_roh.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figure saved: {plot_path}")
plt.show()

## Additional checks: extra variants and trend differences

In [ ]:
print("\n" + "=" * 80)
print("ADDITIONAL CHECK: lemma-only extras and trend differences")
print("=" * 80)

# Which raw suffixes appear only because lemma reduction maps them to the three terms?
raw_exact_suffixes = set(RAW_SUFFIX_MAP.keys())
new_in_reduction = sorted(set(lemma_only["suffix"].str.lower().str.strip().dropna().unique()) - raw_exact_suffixes)

print(f"\nUnique exact raw suffixes: {len(raw_exact_suffixes)}")
print(f"Unique extra raw suffixes added by reduction: {len(new_in_reduction)}")
print(f"Extra mentions added by reduction: {len(lemma_only)}")

if new_in_reduction:
    extra_counts = (
        lemma_only[lemma_only["suffix"].str.lower().str.strip().isin(new_in_reduction)]["suffix"]
        .str.lower()
        .str.strip()
        .value_counts()
        .head(20)
)
    print("\nTop-20 extra raw suffixes captured by reduction:")
    print(extra_counts.to_string())
else:
    print("No extra suffixes found.")

print("\nMean absolute quarterly deviation (pp) per term:")
print(mean_abs_diff.to_string())
print("\nCorrelation of quarterly shares per term:")
print(trend_corr.to_string())
print("\n" + "=" * 80)

## Conclusion

In [ ]:
print("\n" + "=" * 72)
print("CONCLUSION: applied reduction vs. raw exact suffixes")
print("=" * 72)

total_reduced = len(df_reduced)
total_raw = len(df_raw)
difference = total_reduced - total_raw
pct_increase = (difference / total_raw * 100) if total_raw > 0 else 0

print("\n1) DEFINITIONS:")
print("   - Applied reduction: suffix_lemma -> wandel, krise, schutz")
print("   - Raw exact: suffix in {wandel, wandels, krise, krisen, schutz, schutzes}")

print("\n2) COVERAGE (context-only):")
print(f"   - Applied reduction: {total_reduced:,} mentions")
print(f"   - Raw exact: {total_raw:,} mentions")
print(f"   - Difference: {difference:,} ({pct_increase:.1f}%)")

print("\n3) RELATIVE OVERALL DISTRIBUTION (diff in percentage points):")
for term in ["Klimawandel", "Klimakrise", "Klimaschutz"]:
    diff = comparison.loc[term, "Diff (pp)"]
    print(f"   - {term}: {diff:+.2f} pp")

print("\n4) QUARTERLY TREND SIMILARITY:")
for term in ["Klimawandel", "Klimakrise", "Klimaschutz"]:
    mad = mean_abs_diff.get(term, float("nan"))
    corr = trend_corr.get(term, float("nan"))
    print(f"   - {term}: mean deviation {mad:.2f} pp, correlation {corr:.3f}")

if (trend_corr.fillna(0) >= 0.9).all():
    print("\nAssessment: Relative trends remain very similar (robust).")
elif (trend_corr.fillna(0) >= 0.8).all():
    print("\nAssessment: Relative trends remain largely similar (robust).")
else:
    print("\nAssessment: Noticeable trend deviations between variants.")

print("\n5) DATA QUALITY:")
print(f"   - Time filter: {analysis_start.date()} to {analysis_end_label}")
print(f"   - Newspapers in sample: {df['newspaper_name'].nunique()}")
print("   - Scraper change on 2022-04-21 accounted for")

print("\n" + "=" * 72)